In [26]:
import os
import io
from dotenv import load_dotenv
import base64
import numpy as np
import torch
from PIL import Image
import fitz
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema.messages import HumanMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
# Clip Model

load_dotenv()

# set up the environment
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# Initialize the Clip Model for unified embeddings
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

clip_model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [28]:
# Embedding functions

def embed_image(image_data):
    '''Embed image using CLIP'''
    
    # Load image
    if isinstance(image_data, str): # if the input is a file path i.e. if os.path.isfile(image_data):
        image = Image.open(image_data).convert('RGB') # Load image from file path in RGB mode
        
    elif isinstance(image_data, bytes): # if the input is bytes i.e. image uploaded via web form
        image = Image.open(io.BytesIO(image_data)).convert('RGB') # Load image from bytes in RGB mode
        
    else:
        image = image_data # assume input is already a PIL Image object
        
        
    # Preprocess image
    inputs = clip_processor(images=image, return_tensors='pt') # Preprocess image for CLIP model in PyTorch format

    # Get image embeddings
    with torch.no_grad():
        image_data = clip_model.get_image_features(**inputs) # Get image features from CLIP model
        
        # Normalize image embeddings to unit length i.e. L2 normalization
        image_data = image_data / image_data.norm(dim=-1, keepdim=True) # 
        
        return image_data.squeeze().numpy() # Return image embeddings as numpy array
    
# Embedding function for text    
def embed_text(text):
    '''Embed text using CLIP'''
    
    # Preprocess text to be used with CLIP
    inputs = clip_processor(text=[text], return_tensors='pt', padding=True) # Preprocess text for CLIP model in PyTorch format

    # Get text embeddings 
    with torch.no_grad():
        text_data = clip_model.get_text_features(**inputs) # Get text features from CLIP model
        
        # Normalize text embeddings to unit length i.e. L2 normalization
        text_data = text_data / text_data.norm(dim=-1, keepdim=True) 
        
        return text_data.squeeze().numpy() # Return text embeddings as numpy array

In [29]:
# Process PDF
pdf_path = 'multimodal_sample.pdf' # Path to your PDF file

doc = fitz.open(pdf_path) # Open PDF file using PyMuPDF

# Storage for documents and embeddings
documents = [] # List to store Document objects
embeddings = [] # List to store embeddings
image_data_store = {} # Dictionary to store image data with keys as image identifiers

# Text slitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

In [30]:
doc

Document('multimodal_sample.pdf')

In [31]:
for i, page in enumerate(doc):
    # --- Process text ---
    text = page.get_text()
    if text.strip():
        # Keep metadata simple for the splitter to inherit
        temp_doc = Document(page_content=text, metadata={'page': i, 'type': 'text'})
        text_chunks = text_splitter.split_documents([temp_doc])
        
        for chunk in text_chunks:
            # Estimate tokens
            estimated_tokens = len(chunk.page_content) // 4

            if estimated_tokens <= 77:
                text_embedding = embed_text(chunk.page_content)
                # chunk already has metadata={'page': i, 'type': 'text'} from the splitter
                documents.append(chunk) 
                embeddings.append(text_embedding)
                
    # --- Process images ---
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            
            pil_image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
            image_id = f'page_{i}_img_{img_index}'
            
            # Store base64 for later
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG") 
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # CLIP Embedding
            image_embedding = embed_image(pil_image)
            embeddings.append(image_embedding)
            
            # UPDATED: Create Document object to match your desired output
            image_doc = Document(
                page_content=f'[Image: {image_id}]', 
                metadata={'page': i, 'type': 'image', 'image_id': image_id}
            )
            documents.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

# To verify the output format:
print(documents)

[Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]


In [33]:
documents

[Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [34]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(embeddings)
embeddings_array

array([[ 1.73234027e-02, -1.32769002e-02, -2.42703408e-02,
         3.84329632e-02,  1.08081596e-02, -5.86464405e-02,
         2.29126029e-02,  6.74869046e-02,  4.06084396e-02,
         1.29936042e-03,  4.29866090e-03, -2.49144738e-03,
        -5.68564199e-02, -2.10358221e-02,  3.03447861e-02,
        -1.62468012e-02,  2.49861125e-02, -4.18760069e-03,
        -1.45019414e-02, -1.03967767e-02,  1.44381970e-02,
         9.27796215e-03,  3.10424771e-02, -2.87955184e-03,
         1.45214256e-02,  3.06813642e-02, -6.94430843e-02,
         4.24876297e-03,  2.95645241e-02, -3.61430682e-02,
         9.63677559e-03,  1.62329171e-02,  2.33893897e-02,
         1.11387772e-02,  4.35179286e-03,  2.09863228e-03,
         7.63552845e-04,  8.88350885e-03,  2.59088050e-03,
        -1.11683160e-01, -2.35922467e-02, -2.17379071e-03,
        -1.86665487e-02,  2.37778313e-02, -1.48263983e-02,
        -2.55657453e-02, -2.96970387e-03, -8.63732770e-03,
         7.52160186e-03, -1.19852126e-02,  3.99882719e-0

In [36]:
(documents,embeddings_array)

([Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 array([[ 1.73234027e-02, -1.32769002e-02, -2.42703408e-02,
          3.84329632e-02,  1.08081596e-02, -5.86464405e-02,
          2.29126029e-02,  6.74869046e-02,  4.06084396e-02,
          1.29936042e-03,  4.29866090e-03, -2.49144738e-03,
         -5.68564199e-02, -2.10358221e-02,  3.03447861e-02,
         -1.62468012e-02,  2.49861125e-02, -4.18760069e-03,
         -1.45019414e-02, -1.03967767e-02,  1.44381970e-02,
          9.27796215e-03,  3.10424771e-02, -2.87955184e-03,
          1.45214256e-02,  3.06813642e-02, -6.94430843e-02,
          4.24876297e-03,  2.95645241e-02, -3.61430682e-02,
          9.63677559e-03,  1.62329171e-02,  2.33893897e-02,
          1.11387772e-02,  4.35179286e-03,  2.09863228e-03,
          7.63552845e-04,  8.88350885e-03,  2.59088050e-03,
         -1.11683160e-01, -2.35922467e-02, -2.17379071e-03,
         -1.86665487e-02,  2.37778313e-02,